## Extracting Reference Data from REST API Endpoints

### Understanding the Source
![Data_source.png](https://raw.githubusercontent.com/bijay-0011/images/refs/heads/main/3.png)

Recall from our Extraction Strategy Matrix that our fourth and final Bronze source is a set of **REST API endpoints** — `translation` and `geolocation` — served locally by a FastAPI service running inside Docker.

These represent **rarely-changing reference data**, unlike our OLTP tables or the streaming payment/review drops. Because of that, our strategy here is simpler than watermarking or `xmin` tracking:

| Endpoint | Data Shape | Strategy | Bronze Target Path |
| :--- | :--- | :--- | :--- |
| `/translation` | Small, single response | **Full Dump** — fetch once, skip if already extracted | `raw-bronze/api/api_endpoint=translations/` |
| `/geolocation` | Large, paginated | **Paginated Harvest** — follow `next_url` cursors until exhausted | `raw-bronze/api/api_endpoint=geolocation/` |

Since this reference data barely changes, we don't need a watermark or `xmin` state file at all. Instead, we use a much simpler idempotency check: **does the target Parquet file already exist in MinIO?** If yes, skip extraction entirely — no point re-fetching data that isn't going to look any different.

### A Note on Local Docker Endpoints

Since our FastAPI service is running inside Docker on your machine, `CONFIG["api"]["fastapi_url"]` should resolve to something like `http://localhost:8000` (or the mapped port for your container) — **not** a public internet address. Before writing any extraction code, let's just confirm the service is reachable.

In [40]:
# 1. Enable autoreload first
%load_ext autoreload
%autoreload 2

# Import necessary libraries
import os
import requests
import pandas as pd
import time
import datetime
import io

import sys
sys.path.append(os.path.dirname(os.getcwd()))  # Add the parent directory to the sys.path to import config_loader.py. Since the config_loader.py is in the parent directory, we need to add the parent directory to the sys.path so that Python can find it.
from config_loader import CONFIG # Make sure you can access CONFIG variable .

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [41]:
import requests

api_base_url = CONFIG["api"]["fastapi_url"]
print(f"API Base URL: {api_base_url}")

API Base URL: http://localhost:8001


In [42]:
import requests

api_base_url = CONFIG["api"]["fastapi_url"]
print(f"API Base URL: {api_base_url}")

# Quick reachability check
resp = requests.get(f"{api_base_url}/translation", timeout=5)
print(f"Status: {resp.status_code}")


API Base URL: http://localhost:8001
Status: 200


> Note: A `404` on the root path `http://localhost:8001` is expected — FastAPI doesn't define a route at `/` unless you add one explicitly; it just confirms the service is up and responding, not that something's broken.

## `requests.get()` — Retrieving Data from a Server

`requests.get(url)` sends an HTTP GET request to fetch data from a server. 
You configure the request using **arguments**, and read the result using 
**attributes/methods on the response object** it returns.

### Arguments passed INTO `requests.get()`

| Argument | Example | Description |
|---|---|---|
| url | requests.get('https://api.example.com') | The endpoint to send the request to (required) |
| params | params={'q': 'python'} | Query parameters appended to the URL |
| headers | headers={'Authorization': 'Bearer TOKEN'} | Custom headers sent with the request |
| timeout | timeout=5 | Max seconds to wait before giving up |

### Attributes/Methods available ON the `response` object

| Attribute/Method | Example | Description |
|---|---|---|
| status_code | response.status_code | HTTP status code (200, 404, etc.) |
| text | response.text | Response body as a string |
| json() | response.json() | Parses response body as JSON |
| headers | response.headers | Dictionary of response headers |
| content | response.content | Response body as raw bytes |
| url | response.url | Final URL after any redirects |
| ok | response.ok | True if status code < 400 |
| raise_for_status() | response.raise_for_status() | Raises `requests.exceptions.HTTPError` if status code is 4xx or 5xx |
| cookies | response.cookies | Cookies sent back by the server |

> **Note:** Think of it as a two-step flow — **you send** configuration 
> *into* `get()` (url, params, headers, timeout), and the server sends 
> back a `response` object *from which* you read the result (status_code, 
> text, json(), etc.). Arguments go in, attributes come out.

### Step 1: Exploring the Translation Endpoint

The `/translation` endpoint is small and returns its entire dataset in a single response — no pagination needed. Let's hit it directly and inspect the shape of the response before writing any extraction logic.

In [43]:
translation_url = f"{api_base_url}/translation"

response = requests.get(translation_url,timeout=10)
response.raise_for_status()

data = response.json()
print(f"Type: {type(data)}")
print(f"Number of records: {len(data)}")
data[:3]  # peek at the first few records
# list of json in dictionaries

Type: <class 'list'>
Number of records: 71


[{'product_category_name': 'beleza_saude',
  'product_category_name_english': 'health_beauty'},
 {'product_category_name': 'informatica_acessorios',
  'product_category_name_english': 'computers_accessories'},
 {'product_category_name': 'automotivo',
  'product_category_name_english': 'auto'}]

In [44]:
# Convert your list of dict in to dataframe
df_translation = pd.DataFrame(data)
print(f"Shape: {df_translation.shape}")
display(df_translation.head())

Shape: (71, 2)


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


### Step 2: Exploring the Geolocation Endpoint (Paginated)

Unlike `/translation`, the `/geolocation` endpoint is large and can't be returned in a single response. Instead, it uses **cursor-based pagination** — each response contains a page of `data`, plus a `next_url` pointing to the next page. When `next_url` is `null`, we've reached the end.

Let's inspect just the *first page* before writing a loop.

In [45]:
geolocation_url = f"{api_base_url}/geolocation"

response = requests.get(url=geolocation_url,timeout=10)
response.raise_for_status()

payload = response.json()
payload.keys()
payload.get("data")  # list of json in dictionaries

[{'geolocation_zip_code_prefix': 1037,
  'geolocation_lat': -23.54562128115268,
  'geolocation_lng': -46.63929204800168,
  'geolocation_city': 'sao paulo',
  'geolocation_state': 'SP'},
 {'geolocation_zip_code_prefix': 1046,
  'geolocation_lat': -23.546081127035535,
  'geolocation_lng': -46.64482029837157,
  'geolocation_city': 'sao paulo',
  'geolocation_state': 'SP'},
 {'geolocation_zip_code_prefix': 1046,
  'geolocation_lat': -23.54612896641469,
  'geolocation_lng': -46.64295148361138,
  'geolocation_city': 'sao paulo',
  'geolocation_state': 'SP'},
 {'geolocation_zip_code_prefix': 1041,
  'geolocation_lat': -23.5443921648681,
  'geolocation_lng': -46.63949930627844,
  'geolocation_city': 'sao paulo',
  'geolocation_state': 'SP'},
 {'geolocation_zip_code_prefix': 1035,
  'geolocation_lat': -23.541577961711493,
  'geolocation_lng': -46.64160722329613,
  'geolocation_city': 'sao paulo',
  'geolocation_state': 'SP'},
 {'geolocation_zip_code_prefix': 1012,
  'geolocation_lat': -23.54776

In [46]:
print(f"Top-level keys: {list(payload.keys())}")
print(f"Records on this page: {len(payload.get('data', []))}")
print(f"Next URL: {payload.get('next_url')}")

Top-level keys: ['page', 'limit', 'count', 'next_url', 'data']
Records on this page: 1000
Next URL: http://localhost:8001/geolocation?page=2&limit=1000


### Step 5: Following the Pagination Cursor

Now let's follow `next_url` in a loop, accumulating records until there are no more pages. We add a small `time.sleep()` between requests as a courtesy throttle — even though this is a local Docker service, it's good practice for any real API you'd extract from.

We'll test this on a **small page limit** first, just to make sure the loop terminates correctly before running it against the full dataset.

In [47]:
import time
current_url = geolocation_url

all_records = []
pages_fetched = 0

while current_url:
    response = requests.get(current_url,timeout=10)
    response.raise_for_status()

    payload = response.json()
    page_data = payload.get("data",[])
    all_records.extend(page_data)
    pages_fetched += 1

    current_url = payload.get('next_url')
    time.sleep(0.05)

    if pages_fetched >= 5:  # safety cap for this manual test
        print("Stopping early — this is just a test run.")
        break

print(f"Pages fetched: {pages_fetched}")
print(f"Total records accumulated: {len(all_records)}")

Stopping early — this is just a test run.
Pages fetched: 5
Total records accumulated: 5000


In [48]:
df_geo_sample = pd.DataFrame(all_records)
display(df_geo_sample.head())
#db hash

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


### Step 3: Idempotency — Checking If We Already Extracted This

Since translation and geolocation data rarely changes, re-fetching and re-uploading it on every pipeline run would just clutter our bucket with identical duplicate files. Instead of a `state.json` watermark file, we use something much lighter: a **`HEAD` request** against S3 to check if our target Parquet file already exists.

`s3.head_object(Bucket, Key)` is a cheap, metadata-only call — it doesn't download the object, it just asks "does this key exist?" If it doesn't, S3/MinIO raises a `ClientError` with code `404`, which we can catch. For any changes that might occur we can just add a flag to code to update if any changes. 

In [49]:
import boto3
from botocore.exceptions import ClientError

s3 = boto3.client(
    "s3",
    endpoint_url=CONFIG["storage"]["minio_endpoint"],
    aws_access_key_id=CONFIG["storage"]["minio_access_key"],
    aws_secret_access_key=CONFIG["storage"]["minio_secret_key"],
)

In [50]:
s3_key_translation = "api_test/api_endpoint=translations/translations_latest.parquet"

bucket_name = CONFIG["storage"]["bronze_bucket"]

try:
    s3.head_object(Bucket=bucket_name, Key=s3_key_translation)
    print(f"✅Already exists at s3://{bucket_name}/{s3_key_translation}")
except ClientError as e:
    if e.response["Error"]["Code"] == "404":
        print(f"❌ {s3_key_translation} does not exist yet — safe to extract.")
    else:
        raise e



✅Already exists at s3://raw-bronze/api_test/api_endpoint=translations/translations_latest.parquet


Let's create a function to check if the object is already exist in the bucket

In [51]:
def object_exits_in_s3(s3_key:str=None) -> bool:
    try:
        # Check the object
        s3.head_object(Bucket=bucket_name, Key=s3_key)
        print(f"✅ s3://{bucket_name}/{s3_key} already exits.")
        return True
    except ClientError as e:
        if e.response['Error']['Code'] == '404':
            print(f"❌ {s3_key} does not exist yet — safe to extract.")
            return False
        print(f"Unexpected error checking existence of '{s3_key}': {e}")
        raise

object_exits_in_s3("api_test/api_endpoint=translations/translations_latest.parquet")

✅ s3://raw-bronze/api_test/api_endpoint=translations/translations_latest.parquet already exits.


True

### Step 3: Putting It Together — Translation Extraction

Now let's combine the fetch → convert → upload steps into a single manual run, using the existence check as our skip condition.

For versioning, we use a `force_refresh` flag. By default, if `translations_latest.parquet` already exists, we skip extraction. But when `force_refresh=True`:

1. If a `translations_latest.parquet` already exists, **archive it first** — copy it to a new key like `translations_old_20260811_143022.parquet` in the same directory, then delete the original. S3 has no native "rename," so this copy-then-delete is the standard way to simulate one.
2. Fetch fresh data from the API.
3. Upload it to `translations_latest.parquet` — so that key always points to the most recent extract, while every previous version sits alongside it under an `_old_` name.

This gives us one predictable "latest" pointer for downstream consumers, plus a full history for anyone who needs to look back.

Since we will use this archiving logic for other API as well lets define a function to copy-delete the existing object 

In [52]:
key = "api_test/api_endpoint=translations/translations_latest.parquet"

In [53]:
# Build the archived key: translations_latest.parquet -> translations_old_20260811_143022.parquet
directory, filename = key.rsplit("/", 1)
print(f"Directory: {directory}, Filename: {filename}")
name, ext = filename.rsplit(".", 1)
print(f"Name: {name}, Extension: {ext}")
name = name.replace("_latest", "")  # strip "_latest" before appending "_old_<timestamp>"   


Directory: api_test/api_endpoint=translations, Filename: translations_latest.parquet
Name: translations_latest, Extension: parquet


In [54]:
from datetime import datetime
def archive_existing_object(bucket: str, key: str):
    """
    If an object exists at `key`, copy it to a new key with an '_old_<timestamp>'
    suffix inserted before the extension, then delete the original.
    """
    # Build the archived key: translations_latest.parquet -> translations_old_20260811_143022.parquet
    directory, filename = key.rsplit("/", 1)
    name, ext = filename.rsplit(".", 1)
    name = name.replace("_latest", "")  # strip "_latest" before appending "_old_<timestamp>"

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    archived_key = f"{directory}/{name}_old_{timestamp}.{ext}"

    # Copy then delete = simulate a "rename"
    # s3.delete_object(
    #     Bucket=bucket, 
    #     Key=archived_key,
    #     CopySource={
    #         'Bucket': bucket, 
    #         'Key': key
    #     })
    # s3.delete_object(Bucket=bucket, Key=key)
    try:
        s3.copy_object(Bucket=bucket, CopySource={'Bucket': bucket, 'Key': key}, Key=archived_key)
        s3.delete_object(Bucket=bucket, Key=key)
    except ClientError as e:
        print(f"Error archiving object '{key}': {e}")
        raise

    print(f"📦 Archived existing '{key}' ➔ '{archived_key}'")


# Test it
# archive_existing_object(bucket_name, "api/api_endpoint=translations/translations_latest.parquet")

In [55]:
def run_translation_extraction(force_refresh: bool = False):
    s3_key = "api_test/api_endpoint=translations/translations_latest.parquet"

    # 1. Skip check and upload — bypassed entirely if force_refresh is True
    if not force_refresh and object_exits_in_s3(s3_key):
        print(f"Skipping extraction since s3://{bucket_name}/{s3_key} already exists and force_refresh is False.")
        return
    # if object_exits_in_s3(s3_key) and not force_refresh:
    #     print(f"Skipping extraction since s3://{bucket_name}/{s3_key} already exists and force_refresh is False.")
    #     return

    if force_refresh:
        archive_existing_object(bucket_name, s3_key)
    # 2. Fetch
    response = requests.get(translation_url, timeout=10)
    response.raise_for_status()
    df = pd.DataFrame(response.json())

    if df.empty:
        print(" Translation data is empty. Skipping upload.")
        return
    
    # 4. Upload to bucket
    buffer = io.BytesIO()
    df.to_parquet(buffer, index = False, engine = 'pyarrow')

    s3.upload_fileobj(buffer, bucket_name, s3_key)
    print(f"✅ Extracted {len(df)} translation records ➔ s3://{bucket_name}/{s3_key}")


# Test both paths
#run_translation_extraction() 
run_translation_extraction(force_refresh=True)                 # normal run — skip if already exists
# run_translation_extraction(force_refresh=True) # force a new versioned extract

📦 Archived existing 'api_test/api_endpoint=translations/translations_latest.parquet' ➔ 'api_test/api_endpoint=translations/translations_old_20260903_071356.parquet'
✅ Extracted 71 translation records ➔ s3://raw-bronze/api_test/api_endpoint=translations/translations_latest.parquet


### Step 4: Applying the Same Archiving Pattern to Geolocation

Same flow — archive the existing `geolocation_latest.parquet` if `force_refresh=True`, then re-paginate and overwrite `geolocation_latest.parquet` with the fresh pull.

In [56]:
def run_geolocation_extraction(force_refresh: bool = False):
    s3_key = "api_test/api_endpoint=geolocation/geolocation_latest.parquet"

    if not force_refresh and object_exits_in_s3(s3_key):
        print("Nothing Change...")
        return

    if force_refresh:
        archive_existing_object(bucket=bucket_name, key= s3_key)

    # Fetch via pagination
    current_url = f"{api_base_url}/geolocation?page=1&limit=100000"
    all_records = []
    session = requests.Session()

    while current_url:
        response = session.get(translation_url, timeout=10)
        response.raise_for_status()

        payload = pd.DataFrame(response.json())
        page_data = payload.get("data", [])
        all_records.extend(page_data)

        current_url = payload.get('next_url')
        time.sleep(0.05)

    df = pd.DataFrame(all_records)

    if df.empty:
        print(" No geolocation records retrieved. Skipping upload.")
        return

    # Upload file to Bucket
    buffer = io.BytesIO()
    df.to_parquet(buffer, index= False, engine='pyarrow')
    buffer.seek(0)

    s3.upload_fileobj(buffer, bucket_name, s3_key)
    print(f"✅ Uploaded total {len(df)} geolocation records ➔ s3://{bucket_name}/{s3_key}")

# run_geolocation_extraction()
run_geolocation_extraction()

✅ s3://raw-bronze/api_test/api_endpoint=geolocation/geolocation_latest.parquet already exits.
Nothing Change...


### Step 5: Formalizing Into `api_extractor.py`

Let's fold the archive-then-overwrite pattern into our extractor classes. Since **both** `TranslationAPIExtractor` and `GeolocationAPIExtractor` need identical archiving behavior, `_archive_existing_object(key)` belongs on `BaseAPIExtractor` — not duplicated in each subclass.

Fill in the blanks below using the logic we tested above.

In [57]:
import pandas as pd

In [58]:
from api_extractor import TranslationAPIExtractor, GeolocationAPIExtractor

In [59]:
trans_extractor = TranslationAPIExtractor()
trans_extractor.run(force_refresh=True)

2026-09-03 07:13:56 [INFO] extractor.base_extractor: Bucket 'raw-bronze' already exists.
2026-09-03 07:13:56 [INFO] api_extractor: --- [API Extractor] Fetching Translation Data ---
2026-09-03 07:13:56 [INFO] api_extractor: 📦 Archived existing 'api/api_endpoint=translations/translations_latest.parquet' ➔ 'api/api_endpoint=translations/translations_old_20260903_071356.parquet'
2026-09-03 07:13:56 [INFO] api_extractor: ✅ Extracted 71 translation records ➔ s3://raw-bronze/api/api_endpoint=translations/translations_latest.parquet


In [60]:
geo_extractor = GeolocationAPIExtractor()
geo_extractor.run(force_refresh=False)

2026-09-03 07:13:56 [INFO] extractor.base_extractor: Bucket 'raw-bronze' already exists.
2026-09-03 07:13:56 [INFO] api_extractor: --- [API Extractor] Fetching Geolocation Data ---
2026-09-03 07:13:56 [INFO] api_extractor: ⏭️  'api/api_endpoint=geolocation/geolocation_latest.parquet' already exists. Skipping. (pass force_refresh=True to override)


## Extracting Streamed Data from an External S3 Bucket (Payments & Reviews)

### Understanding the Source

Recall from our Extraction Strategy Matrix that payment and review events arrive as **append-only Parquet file drops** in an external MinIO bucket (`pay-review`) — separate from our `raw-bronze` data lake bucket. This simulates a common real-world pattern: a third-party system (a payment processor, a review platform) drops new files into a shared bucket on its own schedule, and it's our job to notice new arrivals and pull them into our lake.

This is fundamentally different from our Postgres or API sources:
- **No SQL, no HTTP** — everything is an S3-to-S3 operation.
- **No transformation needed** — the files are already Parquet. We're not converting anything; we're just *relocating* files from one bucket to another.
- **New files, not new rows** — our "delta" here is *new objects* landing in the source bucket, not new rows in a table.

#### Extraction Strategy

Since MinIO objects carry a `LastModified` timestamp automatically, we can reuse the exact same **watermarking** idea from our Postgres incremental extractor — except instead of comparing a `timestamp` *column*, we compare each object's `LastModified` *metadata*.

1. List all objects in the source bucket under a given prefix (e.g. `order_payments/`).
2. Compare each object's `LastModified` timestamp against our last saved watermark.
3. Any object newer than the watermark is a "new file" — copy it, server-side, directly into `raw-bronze`.
4. Update the watermark to the newest `LastModified` timestamp seen in this batch.

#### Why "Server-Side Copy"?

Since both the source (`pay-review`) and destination (`raw-bronze`) live on the **same MinIO server**, we don't need to download the file to our machine and re-upload it. `s3.copy_object()` tells MinIO to copy the object internally, without the bytes ever passing through our Python process. This is much faster and uses far less bandwidth than a download-then-upload round trip.

In [61]:
source_bucket = "pay-review"
prefix = "order_payments/"

response = s3.list_objects_v2(Bucket=source_bucket, Prefix=prefix)
for obj in response.get("Contents", [])[:10]:
    print(f"{obj['Key']}  |  LastModified: {obj['LastModified']}")
pd.DataFrame(response['Contents'])

order_payments/batch_date=2026-08-06/batch_083200_835051.parquet  |  LastModified: 2026-08-06 08:32:00.843000+00:00
order_payments/batch_date=2026-08-06/batch_083201_875471.parquet  |  LastModified: 2026-08-06 08:32:01.879000+00:00
order_payments/batch_date=2026-08-06/batch_083202_905932.parquet  |  LastModified: 2026-08-06 08:32:02.910000+00:00
order_payments/batch_date=2026-08-06/batch_083203_935790.parquet  |  LastModified: 2026-08-06 08:32:03.939000+00:00
order_payments/batch_date=2026-08-06/batch_083204_967501.parquet  |  LastModified: 2026-08-06 08:32:04.971000+00:00
order_payments/batch_date=2026-08-06/batch_083206_057305.parquet  |  LastModified: 2026-08-06 08:32:06.061000+00:00
order_payments/batch_date=2026-08-06/batch_083207_088550.parquet  |  LastModified: 2026-08-06 08:32:07.093000+00:00
order_payments/batch_date=2026-08-06/batch_083214_189879.parquet  |  LastModified: 2026-08-06 08:32:14.194000+00:00
order_payments/batch_date=2026-08-06/batch_083215_221157.parquet  |  Las

,Key,LastModified,ETag,Size,StorageClass
0,order_payments/batch_date=2026-08-06/batch_083...,2026-08-06 08:32:00.843000+00:00,"""ddb3085e83d1799c680d57c75d71c2a1""",3964,STANDARD
1,order_payments/batch_date=2026-08-06/batch_083...,2026-08-06 08:32:01.879000+00:00,"""7e4ef1b8ad04fbdab2433af4ad2bda0c""",3846,STANDARD
2,order_payments/batch_date=2026-08-06/batch_083...,2026-08-06 08:32:02.910000+00:00,"""550c443f06a643216910c3a3ef21cae6""",3929,STANDARD
3,order_payments/batch_date=2026-08-06/batch_083...,2026-08-06 08:32:03.939000+00:00,"""2099e434df3d92a737b23c9fde928c3e""",3898,STANDARD
4,order_payments/batch_date=2026-08-06/batch_083...,2026-08-06 08:32:04.971000+00:00,"""2075555dd60768520e527cd21f839f9d""",4058,STANDARD
...,...,...,...,...,...
598,order_payments/batch_date=2026-08-26/batch_082...,2026-08-26 08:23:41.796000+00:00,"""f1f5cebbf4d136838e35e8673ddcc1b9""",4195,STANDARD
599,order_payments/batch_date=2026-08-26/batch_082...,2026-08-26 08:23:42.918000+00:00,"""cfe40005d062e29624f8cc9d240f4ea1""",4190,STANDARD
600,order_payments/batch_date=2026-08-26/batch_082...,2026-08-26 08:23:44.040000+00:00,"""372d70fcddf254bdbec3fd270af718d9""",3937,STANDARD
601,order_payments/batch_date=2026-08-26/batch_082...,2026-08-26 08:23:45.146000+00:00,"""35606183ca2d41e65255e0279fdc2ab3""",4085,STANDARD


### Step 1: Reading the Watermark State

Just like our Postgres incremental extractor, we'll persist a watermark per source table under `metadata/`. The only difference is the key prefix — here we use `external_table={table}` to distinguish this source's state files from Postgres's.

In [62]:
table = "order_payments"
state_key = f"metadata_test/external_table={table}/state.json"

In [63]:
import json
from botocore.exceptions import ClientError

table = "order_payments"
state_key = f"metadata_test/external_table={table}/state.json"

try:
    response = s3.get_object(Bucket=bucket_name, Key=state_key)
    state_data = json.loads(response["Body"].read().decode("utf-8"))
    last_synced_ts = state_data.get("last_synced_timestamp", "1970-01-01T00:00:00+00:00")
    print(f"Found existing watermark: {last_synced_ts}")
except ClientError as e:
    if e.response["Error"]["Code"] == "NoSuchKey":
        last_synced_ts = "1970-01-01T00:00:00+00:00"
        print(f"No previous watermark found. Defaulting to: {last_synced_ts}")
    else:
        raise e

No previous watermark found. Defaulting to: 1970-01-01T00:00:00+00:00


In [64]:
def get_last_synced_timestamp(table: str= None):
    state_key = f"metadata/external_table={table}/state.json"
    default_timestamp = "1970-01-01T00:00:00+00:00"
    try:
        response = s3.get_object(Bucket=bucket_name, Key=state_key)
        state_data = json.loads(response["Body"].read().decode("utf-8"))
        last_synced_ts = state_data.get("last_synced_timestamp", "1970-01-01T00:00:00+00:00")
        print(f"Found existing watermark for '{table}': {last_synced_ts}")
        return last_synced_ts
    except ClientError as e:
        if e.response["Error"]["Code"] == "NoSuchKey":
            print(f"No previous state found for '{table}'")
            return default_timestamp
        raise e


### Step 2: Finding New Files Using the Paginator

A source bucket can have more than 1000 objects, which is the hard limit `list_objects_v2` returns in a single call. Just like we saw earlier with our MinIO cheat sheet, we use a **paginator** to safely loop through every object regardless of how many there are.

For each object, we compare its `LastModified` timestamp (converted to an ISO string) against our watermark — anything newer is a "new file" we haven't synced yet.

In [65]:
from pprint import pprint;

In [66]:
paginator = s3.get_paginator("list_objects_v2")
pages = paginator.paginate(Bucket=source_bucket, Prefix=prefix)

new_files = []
max_ts_in_batch = last_synced_ts

for page in pages:
    # pprint(page['Contents'])
    for obj in page.get('Contents', []):
        print(obj)

{'Key': 'order_payments/batch_date=2026-08-06/batch_083200_835051.parquet', 'LastModified': datetime.datetime(2026, 8, 6, 8, 32, 0, 843000, tzinfo=tzlocal()), 'ETag': '"ddb3085e83d1799c680d57c75d71c2a1"', 'Size': 3964, 'StorageClass': 'STANDARD'}
{'Key': 'order_payments/batch_date=2026-08-06/batch_083201_875471.parquet', 'LastModified': datetime.datetime(2026, 8, 6, 8, 32, 1, 879000, tzinfo=tzlocal()), 'ETag': '"7e4ef1b8ad04fbdab2433af4ad2bda0c"', 'Size': 3846, 'StorageClass': 'STANDARD'}
{'Key': 'order_payments/batch_date=2026-08-06/batch_083202_905932.parquet', 'LastModified': datetime.datetime(2026, 8, 6, 8, 32, 2, 910000, tzinfo=tzlocal()), 'ETag': '"550c443f06a643216910c3a3ef21cae6"', 'Size': 3929, 'StorageClass': 'STANDARD'}
{'Key': 'order_payments/batch_date=2026-08-06/batch_083203_935790.parquet', 'LastModified': datetime.datetime(2026, 8, 6, 8, 32, 3, 939000, tzinfo=tzlocal()), 'ETag': '"2099e434df3d92a737b23c9fde928c3e"', 'Size': 3898, 'StorageClass': 'STANDARD'}
{'Key': 'ord

In [67]:
paginator = s3.get_paginator("list_objects_v2")
pages = paginator.paginate(Bucket=source_bucket, Prefix=prefix)

new_files = []
max_ts_in_batch = last_synced_ts

for page in pages:
    for obj in page.get("Contents", []):
        file_modified_ts = obj["LastModified"].isoformat()

        if file_modified_ts > last_synced_ts:
            new_files.append((obj["Key"], file_modified_ts))
            if file_modified_ts > max_ts_in_batch:
                max_ts_in_batch = file_modified_ts

print(f"Found {len(new_files)} new file(s).")
for key, ts in new_files[:5]:
    print(f"  {key}  |  {ts}")

print(f"\nNew watermark would be: {max_ts_in_batch}")

Found 603 new file(s).
  order_payments/batch_date=2026-08-06/batch_083200_835051.parquet  |  2026-08-06T08:32:00.843000+00:00
  order_payments/batch_date=2026-08-06/batch_083201_875471.parquet  |  2026-08-06T08:32:01.879000+00:00
  order_payments/batch_date=2026-08-06/batch_083202_905932.parquet  |  2026-08-06T08:32:02.910000+00:00
  order_payments/batch_date=2026-08-06/batch_083203_935790.parquet  |  2026-08-06T08:32:03.939000+00:00
  order_payments/batch_date=2026-08-06/batch_083204_967501.parquet  |  2026-08-06T08:32:04.971000+00:00

New watermark would be: 2026-08-26T08:23:46.248000+00:00


### Step 3: Server-Side Copy Into `raw-bronze`

Now let's copy each new file directly into our Bronze layer, under `external_s3/stream_event={table}/`. Notice we preserve the original filename — we don't need Hive-style date partitioning here, since the source system itself already names files uniquely (or we could add partitioning later if needed).

`CopySource` tells MinIO *where the file currently lives*; `Key` tells it *where to put the copy*. Both `Bucket` params can point to the same MinIO server since we're just moving data between buckets on one instance.

In [68]:
if new_files:
    source_key, file_ts = new_files[0]
    filename = source_key.split("/")[-1]
    target_key = f"external_s3/stream_event={table}/{filename}"

    s3.copy_object(
        Bucket=bucket_name,
        CopySource={"Bucket": source_bucket, "Key": source_key},
        Key=target_key,
    )
    print(f"Copied: {source_key} ➔ s3://{bucket_name}/{target_key}")
else:
    print("No new files to copy in this test.")

Copied: order_payments/batch_date=2026-08-06/batch_083200_835051.parquet ➔ s3://raw-bronze/external_s3/stream_event=order_payments/batch_083200_835051.parquet


In [69]:
for source_key, file_ts in new_files:
    filename = source_key.split("/")[-1]
    target_key = f"external_s3/stream_event={table}/{filename}"

    s3.copy_object(
        Bucket=bucket_name,
        CopySource={"Bucket": source_bucket, "Key": source_key},
        Key=target_key,
    )

print(f"Copied {len(new_files)} file(s) total.")

Copied 603 file(s) total.


### Step 4: Saving the New Watermark

Once every new file in this batch has been copied, we save the newest `LastModified` timestamp we saw — so next run only picks up files that arrived after this point.

In [70]:
def update_synced_timestamp(table: str, max_timestamp: str):
    state_key = f"metadata/external_table={table}/state.json"
    payload = json.dumps({
        "table": table,
        "last_synced_timestamp": max_timestamp,
        "updated_at": datetime.now().isoformat(),
    })
    
    s3.put_object(
        Bucket= bucket_name,
        Key=state_key,
        Body=payload.encode("utf-8"),
        ContentType="application/json",
    )
    print(f"Updated watermark for '{table}' to {max_timestamp}")

if new_files:
    update_synced_timestamp(table, max_ts_in_batch)

Updated watermark for 'order_payments' to 2026-08-26T08:23:46.248000+00:00


#### Putting it all together

In [72]:
def run_external_s3_sync(source_bucket: str, source_tables: list[str], bronze_bucket: str):
    for table in source_tables:
        print(f"\n--- Checking for new '{table}' files in '{source_bucket}' ---")

        # 1. Get last watermark
        last_synced_ts = get_last_synced_timestamp(table=table)

        # 2. Find new files
        prefix = f"{table}/"
        paginator = s3.get_paginator("list_objects_v2")
        pages = paginator.paginate(Bucket=source_bucket, Prefix=prefix)

        new_files = []
        max_ts_in_batch = last_synced_ts

        for page in pages:
            for obj in page.get("Contents", []):
                file_modified_ts = obj["LastModified"].isoformat()
                if file_modified_ts > last_synced_ts:
                    new_files.append((obj["Key"], file_modified_ts))
                    if file_modified_ts > max_ts_in_batch:
                        max_ts_in_batch = file_modified_ts

        if not new_files:
            print(f"⏭️  No new files in '{table}'. Skipping copy.")
            continue

        print(f"🔥 Found {len(new_files)} new file(s) for '{table}'. Copying...")

        # 3. Server-side copy each new file
        for source_key, file_ts in new_files:
            filename = source_key.split("/")[-1]
            target_key = f"external_s3/stream_event={table}/{filename}"

            # Copy the data to destination
            s3.copy_object(
                Bucket=bronze_bucket,
                CopySource={"Bucket": source_bucket, "Key": source_key},
                Key=target_key,
            )

        # 4. Save new watermark
        update_synced_timestamp(table=table, max_timestamp=max_ts_in_batch)

        print(f"✅ Updated watermark state for '{table}' to {max_ts_in_batch}")


# Test run
run_external_s3_sync(
    source_bucket="pay-review",
    source_tables=["order_payments", "order_reviews"],
    bronze_bucket=bucket_name,
)


--- Checking for new 'order_payments' files in 'pay-review' ---
Found existing watermark for 'order_payments': 2026-08-26T08:23:46.248000+00:00
⏭️  No new files in 'order_payments'. Skipping copy.

--- Checking for new 'order_reviews' files in 'pay-review' ---
No previous state found for 'order_reviews'
🔥 Found 541 new file(s) for 'order_reviews'. Copying...
Updated watermark for 'order_reviews' to 2026-08-26T08:23:45.163000+00:00
✅ Updated watermark state for 'order_reviews' to 2026-08-26T08:23:45.163000+00:00


### Formalizing into s3_extractor.py

In [76]:
from s3_extractor import ExternalS3Extractor
s3_extractor = ExternalS3Extractor()
s3_extractor.run()

2026-09-03 07:17:28 [INFO] extractor.base_extractor: Bucket 'raw-bronze' already exists.
2026-09-03 07:17:28 [INFO] s3_extractor: --- Checking for new 'order_payments' files in 'pay-review' ---


2026-09-03 07:17:28 [INFO] s3_extractor: ⏭️  No new files in 'order_payments'. Skipping copy.
2026-09-03 07:17:28 [INFO] s3_extractor: --- Checking for new 'order_reviews' files in 'pay-review' ---
2026-09-03 07:17:28 [INFO] s3_extractor: ⏭️  No new files in 'order_reviews'. Skipping copy.
